# 🎛️ Generation Control Parameters

**Day 4 — Notebook 3 of 4 | Estimated Time: 25 minutes**

---

## What You'll Learn
- How `temperature`, `top_k`, and `top_p` control randomness in generation
- When to use high vs. low temperature for different tasks
- How to produce deterministic (reproducible) outputs
- Practical experiments to build intuition for each parameter

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0"

In [ ]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from IPython.display import Markdown

client = genai.Client(api_key=get_api_key())
MODEL_ID = "gemini-2.5-flash"
print("✅ Ready!")

---

## 1. How LLM Sampling Works

When an LLM generates a response, it doesn't pick the single "best" next word — it produces a **probability distribution** over all possible tokens and **samples** from it.

```
Prompt: "The capital of France is..."

Token probabilities:
  "Paris"    → 92.3%
  "Lyon"     →  2.1%
  "London"   →  1.4%
  "a"        →  0.8%
  ... others →  3.4%
```

The three main parameters control **which tokens are eligible** and **how likely** the model is to pick lower-probability options:

| Parameter | What it controls |
|-----------|------------------|
| `temperature` | Sharpness of the distribution (0 = deterministic, 2 = very random) |
| `top_k` | Maximum number of tokens to consider (e.g., top 40 only) |
| `top_p` | Cumulative probability threshold (e.g., only tokens summing to 95% probability) |

---

## 2. Temperature

`temperature` controls how "creative" or "random" the output is.

- **temperature = 0** → nearly deterministic, always picks the most likely token
- **temperature = 0.7** → balanced: coherent but varied
- **temperature = 1.5+** → very creative, sometimes incoherent

In [ ]:
prompt = "Continue this story in one sentence: 'The robot opened its eyes and saw'"

temperatures = [0.0, 0.5, 1.0, 1.5]

for temp in temperatures:
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=temp),
    )
    print(f"🌡️  temperature={temp}: {response.text.strip()}")
    print()

**Observe:**
- Low temperature → safe, predictable completions
- High temperature → unexpected, creative (sometimes strange) completions

---

## 3. Temperature for Reproducibility

For factual tasks (code generation, data extraction, classification), **low temperature** = more consistent results.

In [ ]:
# Run the same factual question 3 times at high temperature
print("=== HIGH TEMPERATURE (1.5) — 3 runs ===")
for i in range(3):
    response = client.models.generate_content(
        model=MODEL_ID,
        contents="What is 17 multiplied by 24? Respond with just the number.",
        config=types.GenerateContentConfig(temperature=1.5),
    )
    print(f"Run {i+1}: {response.text.strip()}")

print()
print("=== LOW TEMPERATURE (0.0) — 3 runs ===")
for i in range(3):
    response = client.models.generate_content(
        model=MODEL_ID,
        contents="What is 17 multiplied by 24? Respond with just the number.",
        config=types.GenerateContentConfig(temperature=0.0),
    )
    print(f"Run {i+1}: {response.text.strip()}")

---

## 4. top_p (Nucleus Sampling)

`top_p` filters the token pool to only the **most likely tokens** that together sum to probability `p`.

- **top_p = 1.0** → consider all tokens
- **top_p = 0.9** → only consider tokens that make up the top 90% of probability mass
- **top_p = 0.1** → very restricted — only the most probable tokens

This prevents the model from picking very unlikely "wild card" tokens while still allowing natural variety.

In [ ]:
prompt = "Write a creative metaphor for the concept of 'time'. Just one sentence."

top_p_values = [0.1, 0.5, 0.9, 1.0]

for top_p in top_p_values:
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=1.0,  # Keep temperature constant to isolate top_p effect
            top_p=top_p,
        ),
    )
    print(f"🎯 top_p={top_p}: {response.text.strip()}")
    print()

---

## 5. top_k

`top_k` limits the token pool to the **top K most probable tokens** at each step.

- **top_k = 1** → always pick the single most likely token (greedy decoding)
- **top_k = 40** → choose from the 40 most likely tokens (Gemini default)
- **top_k = 1000** → very broad pool, close to unconstrained

In [ ]:
prompt = "List 3 unusual uses for a paperclip."

top_k_values = [1, 10, 40, 100]

for top_k in top_k_values:
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=1.0,
            top_k=top_k,
        ),
    )
    print(f"🔢 top_k={top_k}:")
    print(response.text.strip())
    print()

---

## 6. max_output_tokens

Controls the **maximum length** of the response. Useful to prevent runaway outputs and manage cost.

In [ ]:
for max_tokens in [20, 100, 500]:
    response = client.models.generate_content(
        model=MODEL_ID,
        contents="Explain the theory of relativity.",
        config=types.GenerateContentConfig(
            max_output_tokens=max_tokens,
        ),
    )
    word_count = len(response.text.split())
    print(f"🔢 max_output_tokens={max_tokens} → ~{word_count} words")
    print(response.text.strip()[:200], "..." if len(response.text) > 200 else "")
    print()

---

## 7. Practical Configuration Guide

Here are recommended settings for common use cases:

In [ ]:
# Use case configurations
configs = {
    "factual_qa": types.GenerateContentConfig(
        temperature=0.1,
        top_p=0.9,
        top_k=40,
        max_output_tokens=512,
    ),
    "creative_writing": types.GenerateContentConfig(
        temperature=1.2,
        top_p=0.95,
        top_k=100,
        max_output_tokens=1024,
    ),
    "code_generation": types.GenerateContentConfig(
        temperature=0.2,
        top_p=0.95,
        top_k=40,
        max_output_tokens=2048,
    ),
    "brainstorming": types.GenerateContentConfig(
        temperature=1.0,
        top_p=1.0,
        top_k=64,
        max_output_tokens=512,
    ),
}

prompt = "Give me an idea for improving team productivity."

for use_case, config in configs.items():
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=config,
    )
    print(f"🎯 [{use_case}]")
    print(response.text.strip()[:300])
    print()

---

## 🏋️ Exercise 1: Temperature Tuning

Find the right temperature for a **customer support chatbot** that needs to be helpful but not too robotic or too unpredictable.

In [ ]:
# Exercise 1: Temperature tuning for customer support
customer_message = """I ordered a laptop 2 weeks ago and it still hasn't arrived. 
My order number is #ORD-4892. I'm really frustrated. What should I do?"""

system_instruction = """You are a helpful customer support agent for an e-commerce company. 
Be empathetic, professional, and provide actionable steps."""

# TODO: Try temperatures [0.0, 0.3, 0.7, 1.2] and decide which feels best for this use case
# Consider: Is the response empathetic? Is it helpful? Does it vary too much between runs?

temperatures_to_test = [0.0, 0.3, 0.7, 1.2]

for temp in temperatures_to_test:
    pass  # TODO: Generate and print each response

---

## 🏋️ Exercise 2: Deterministic Classification

Build a sentiment classifier that gives **consistent results** across multiple runs.

In [ ]:
# Exercise 2: Consistent sentiment classification
reviews = [
    "This product exceeded all my expectations. Absolutely worth the price!",
    "Terrible quality. Broke after two days. Complete waste of money.",
    "It's okay. Does the job but nothing special.",
]

# TODO: Configure generation parameters so that:
# 1. The same review always gets the same classification across multiple runs
# 2. Output is constrained to only: POSITIVE, NEGATIVE, or NEUTRAL
# Hint: What temperature gives you the most consistent results?

# config = types.GenerateContentConfig(
#     temperature=???,
#     max_output_tokens=???,
# )

# Run classification 3 times for each review and check consistency
for review in reviews:
    results = []
    for _ in range(3):
        pass  # TODO: Classify and collect results
    # print(f"Review: {review[:50]}... → Results: {results}")

---

## Parameter Quick Reference

| Parameter | Range | Low Value | High Value | Best for |
|-----------|-------|-----------|------------|----------|
| `temperature` | 0.0–2.0 | Deterministic | Creative/random | Low: classification, code, facts. High: creative writing, brainstorming |
| `top_p` | 0.0–1.0 | Restrictive | All tokens | 0.9–0.95 for most tasks |
| `top_k` | 1–1000 | Greedy | Very open | 40 is a good default |
| `max_output_tokens` | 1–8192+ | Short | Long | Match expected output length |

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| Gemini API Generation Config | Docs | [ai.google.dev/gemini-api/docs/text-generation](https://ai.google.dev/gemini-api/docs/text-generation) |
| Temperature in LLMs Explained | Blog | [huggingface.co/blog/how-to-generate](https://huggingface.co/blog/how-to-generate) |

---

**Next up:** [04_Safety_Settings_and_Guardrails.ipynb](./04_Safety_Settings_and_Guardrails.ipynb)